<a href="https://colab.research.google.com/github/princeowaowa/python/blob/main/Pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Code to create a sample CSV file in Colab's current workspace
import pandas as pd

# Sample data
data = {
    'Name': [' Alice ', 'Bob', 'Alice', 'Charlie', None],
    'Age': [25, 30, 25, None, 22],
    'Salary': ['50000', '60000', '50000', '70000', '45000']
}

# Create a DataFrame and save it as 'your_file.csv'
sample_df = pd.DataFrame(data)
sample_df.to_csv('your_file.csv', index=False)
print("Successfully created 'your_file.csv' in the Colab workspace!")

Successfully created 'your_file.csv' in the Colab workspace!


In [ ]:
# 1. Read the CSV file
try:
    df = pd.read_csv('your_file.csv')
    print("Original DataFrame Shape:", df.shape)
    print("\nFirst 5 rows:")
    print(df.head())
except FileNotFoundError:
    print("CSV file not found. Creating a sample DataFrame for demonstration...")
    data = {
        'Name': [' Alice ', 'Bob', 'Alice', 'Charlie', None],
        'Age': [25, 30, 25, None, 22],
        'Salary': ['50000', '60000', '50000', '70000', '45000']
    }
    df = pd.DataFrame(data)
    print("Sample DataFrame:")
    print(df)

# 2. Perform basic data cleaning methods

# Remove duplicate rows
df = df.drop_duplicates()

# Strip leading and trailing whitespaces from string columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

# Handle missing values
if 'Age' in df.columns:
    df['Age'] = df['Age'].fillna(df['Age'].median())

# Convert data types
if 'Salary' in df.columns:
    df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')

print("\nCleaned DataFrame Shape:", df.shape)
print("\nCleaned DataFrame:")
print(df)

Index(['Name', 'Age', 'Salary'], dtype='object')

In [8]:
# 1. Read the fresh CSV file again
df_fixed = pd.read_csv('your_file.csv')

# 2. Strip whitespaces FIRST before dropping duplicates
# This ensures ' Alice ' and 'Alice' are recognized as the same value
for col in df_fixed.select_dtypes(include='object').columns:
    df_fixed[col] = df_fixed[col].astype(str).str.strip()

# 3. Handle the 'nan' string converted from NaN values
# We replace the literal string 'nan' back to a proper empty string or a placeholder like 'Unknown'
df_fixed['Name'] = df_fixed['Name'].replace('nan', 'Name not available')

# 4. Fill missing Age values
if 'Age' in df_fixed.columns:
    df_fixed['Age'] = df_fixed['Age'].fillna(df_fixed['Age'].median())

# 5. Convert Salary to numeric
if 'Salary' in df_fixed.columns:
    df_fixed['Salary'] = pd.to_numeric(df_fixed['Salary'], errors='coerce')

# 6. Now drop duplicates (after values have been normalized/cleaned!)
df_fixed = df_fixed.drop_duplicates()

# Display the correctly cleaned DataFrame
print("Correctly Cleaned DataFrame:")
display(df_fixed)

Correctly Cleaned DataFrame:


,Name,Age,Salary
0,Alice,25.0,50000
1,Bob,30.0,60000
3,Charlie,25.0,70000
4,Name not available,22.0,45000


In [13]:
# Method to apply some logic to each row and create new column.
def calculate_bonus(row):
    # Custom logic: if the age is over 25, they get a 10% bonus, otherwise 5%
    if row['Age'] > 25:
        return row['Salary'] * 0.10
    else:
        return row['Salary'] * 0.05

# Apply the function to each row (axis=1 refers to row-by-row operations)
df_fixed['Bonus'] = df_fixed.apply(calculate_bonus, axis=1)

# Display the updated DataFrame
display(df_fixed)

,Name,Age,Salary,Bonus
0,Alice,25.0,50000,2500.0
1,Bob,30.0,60000,6000.0
3,Charlie,25.0,70000,3500.0
4,Name not available,22.0,45000,2250.0


In [15]:
# pandas where clause
import numpy as np

# Example 1: Using numpy.where to create a conditional column
# If Salary is greater than 55000, label as 'High', otherwise 'Standard'
df_fixed['Salary_Class'] = np.where(df_fixed['Salary'] > 55000, 'High', 'Standard')

# Example 2: Using pandas DataFrame.where to replace values conditionally
# df.where replaces values where the condition is FALSE.
# Let's cap any salary above 65000 to exactly 65000
df_fixed['Capped_Salary'] = df_fixed['Salary'].where(df_fixed['Salary'] <= 65000, 65000)

display(df_fixed)

,Name,Age,Salary,Bonus,Salary_Class,Capped_Salary
0,Alice,25.0,50000,2500.0,Standard,50000
1,Bob,30.0,60000,6000.0,High,60000
3,Charlie,25.0,70000,3500.0,High,65000
4,Name not available,22.0,45000,2250.0,Standard,45000


In [16]:
# 1. Append more diverse records to our dataset and recreate the CSV
more_data = {
    'Name': [' Alice ', 'Bob', 'Alice', 'Charlie', None, 'David', 'Eva', 'Frank', 'Grace', 'Henry'],
    'Age': [25, 30, 25, None, 22, 35, 28, 40, None, 50],
    'Salary': ['50000', '60000', '50000', '70000', '45000', '85000', '95000', '62000', '54000', '120000']
}

# Save new data to 'your_file.csv'
pd.DataFrame(more_data).to_csv('your_file.csv', index=False)
print("CSV updated with 10 rows of data!")

CSV updated with 10 rows of data!


In [21]:
# 2. Read the new CSV, perform the cleaning steps we built earlier
df_expanded = pd.read_csv('your_file.csv')
for col in df_expanded.select_dtypes(include='object').columns:
    df_expanded[col] = df_expanded[col].astype(str).str.strip()
df_expanded['Name'] = df_expanded['Name'].replace('nan', 'Name not available')
if 'Age' in df_expanded.columns:
    df_expanded['Age'] = df_expanded['Age'].fillna(df_expanded['Age'].median())
if 'Salary' in df_expanded.columns:
    df_expanded['Salary'] = pd.to_numeric(df_expanded['Salary'], errors='coerce')
df_expanded = df_expanded.drop_duplicates()

# 3. Create a filtering function with different filtering methods
def filter_dataframe(dataframe, min_age=None, min_salary=None, name_search=None):
    filtered_df = dataframe.copy()

    # Method A: Boolean Indexing
    if min_age is not None:
        filtered_df = filtered_df[filtered_df['Age'] >= min_age]

    if min_salary is not None:
        filtered_df = filtered_df[filtered_df['Salary'] >= min_salary]

    # Method B: String matching filter
    if name_search is not None:
        filtered_df = filtered_df[filtered_df['Name'].str.contains(name_search, case=False)]

    return filtered_df

# Let's test the filter method: Age >= 28 and Salary >= 60,000
print("Filtered Dataset (Age >= 28 and Salary >= 60,000):")
results = filter_dataframe(df_expanded, min_age=24, min_salary=50000, name_search='li')
display(results)

Filtered Dataset (Age >= 28 and Salary >= 60,000):


,Name,Age,Salary
0,Alice,25.0,50000
3,Charlie,29.0,70000


In [22]:
# Rename columns in the DataFrame using a dictionary mapping
# syntax: df.rename(columns={'old_name': 'new_name'}, inplace=True)
df_expanded.rename(columns={
    'Name': 'Full_Name',
    'Salary': 'Annual_Salary'
}, inplace=True)

# Display the DataFrame with the updated column headers
display(df_expanded.head())

,Full_Name,Age,Annual_Salary
0,Alice,25.0,50000
1,Bob,30.0,60000
3,Charlie,29.0,70000
4,Name not available,22.0,45000
5,David,35.0,85000


In [27]:
# 1. Get the total number of unique values in the 'Full_Name' column
unique_count = df_expanded['Full_Name'].nunique()
print(f"Total unique names: {unique_count}\n")

# 2. Get the count/frequency of each unique value in the 'Full_Name' column
value_counts = df_expanded['Full_Name'].value_counts()
print("Frequency of each unique name:")
display(value_counts)

# Updated 'Salary' to 'Annual_Salary' to match the renamed column name
sal_counts = df_expanded['Annual_Salary'].value_counts()
print("Frequency of each unique salary:")
display(sal_counts)

Total unique names: 9

Frequency of each unique name:


,count
Full_Name,
Alice,1
Bob,1
Charlie,1
Name not available,1
David,1
Eva,1
Frank,1
Grace,1
Henry,1


Frequency of each unique salary:


,count
Annual_Salary,
50000,1
60000,1
70000,1
45000,1
85000,1
95000,1
62000,1
54000,1
120000,1


In [28]:
# Update Bob's salary to match Alice's (50000)
# df_expanded.loc[row_index, 'column_name'] = new_value
df_expanded.loc[df_expanded['Full_Name'] == 'Bob', 'Annual_Salary'] = 50000

# Let's also make David's salary 50000 to have three rows with the same salary
df_expanded.loc[df_expanded['Full_Name'] == 'David', 'Annual_Salary'] = 50000

# Display the updated frequency of each unique salary value
updated_sal_counts = df_expanded['Annual_Salary'].value_counts()
print("Updated Frequency of each unique salary:")
display(updated_sal_counts)

Updated Frequency of each unique salary:


,count
Annual_Salary,
50000,3
70000,1
45000,1
95000,1
62000,1
54000,1
120000,1


In [29]:
df_expanded

,Full_Name,Age,Annual_Salary
0,Alice,25.0,50000
1,Bob,30.0,50000
3,Charlie,29.0,70000
4,Name not available,22.0,45000
5,David,35.0,50000
6,Eva,28.0,95000
7,Frank,40.0,62000
8,Grace,29.0,54000
9,Henry,50.0,120000


In [12]:
df_fixed.shape


(4, 4)